# Lung CT Classification (IQ-OTH/NCCD)

Fine-tunes an ImageNet-pretrained **ResNet-50** to classify chest CT slices as **Benign**, **Malignant** or **Normal**, using the augmented version of the IQ-OTH/NCCD lung cancer dataset.

> Educational project, not a medical device. Do not use it for diagnosis.

Place the dataset folder next to this notebook (see the README), then run the cells in order.

# Imports

In [ ]:
import os
import random

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from tqdm import tqdm

# Settings

In [ ]:
DATA_DIR = "Augmented IQ-OTHNCCD lung cancer dataset"  # holds 'Benign cases', 'Malignant cases', 'Normal cases'
SEED = 42
ORIGINALS_ONLY = True   # see the note below before changing this
BATCH_SIZE = 32
NUM_EPOCHS = 10
LR = 1e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Data

**Why `ORIGINALS_ONLY`?** About 70% of the files in this dataset are augmented copies (flips, rotations, blur, ...) of the 1,097 original CT slices. If the whole folder is split randomly, copies of the same slice end up in both the training and validation sets, which makes the validation accuracy look much better than it really is.

With `ORIGINALS_ONLY = True` only the original slices are used. They are split 70/15/15 into train/validation/test (stratified by class), and light augmentation is applied on the fly to the training set only. Set it to `False` to use every file; the split will then leak and the scores will be inflated.

One limitation remains: slices from the same patient can still land in different splits, because this version of the dataset has no patient IDs.

In [ ]:
# Files created by augmentation start with one of these names. Everything else is an original slice.
AUG_PREFIXES = ("auto_contrast", "colorjitter", "contour_crop", "gaussian_blur",
                "hist_equalize", "horizontal_flip", "rotate", "sharpen", "vertical_flip")

imagenet_norm = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    imagenet_norm,
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    imagenet_norm,
])

# Two views of the same files: one with training augmentation, one without
train_full = datasets.ImageFolder(root=DATA_DIR, transform=train_tf)
eval_full = datasets.ImageFolder(root=DATA_DIR, transform=eval_tf)
class_names = train_full.classes
targets = np.array(train_full.targets)
paths = [p for p, _ in train_full.samples]

is_original = np.array([not os.path.basename(p).startswith(AUG_PREFIXES) for p in paths])
pool = np.where(is_original)[0] if ORIGINALS_ONLY else np.arange(len(paths))

train_idx, rest_idx = train_test_split(pool, test_size=0.30, stratify=targets[pool], random_state=SEED)
val_idx, test_idx = train_test_split(rest_idx, test_size=0.50, stratify=targets[rest_idx], random_state=SEED)

train_dataset = Subset(train_full, train_idx)
val_dataset = Subset(eval_full, val_idx)
test_dataset = Subset(eval_full, test_idx)

for name, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    print(f"{name:5s}: {len(idx):4d} images", {c: int(n) for c, n in zip(class_names, np.bincount(targets[idx], minlength=len(class_names)))})

g = torch.Generator().manual_seed(SEED)
dataloader_train = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=g)
dataloader_val = DataLoader(val_dataset, batch_size=BATCH_SIZE)
dataloader_test = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# Model

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(class_names))  # replace the 1000-class ImageNet head

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

model = model.to(device)

# Training

In [ ]:
def train_epoch(model, loader, optimizer, criterion, num_epochs, epoch):
    model.train()
    total_loss, total, correct = 0., 0, 0

    pbar = tqdm(loader, desc=f'Epoch: {epoch}/{num_epochs}')

    for img, labels in pbar:
        img, labels = img.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(img)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total += labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total_loss += loss.item() * labels.size(0)

        pbar.set_postfix(acc=f'{100 * correct / total:.2f}%', loss=f'{total_loss / total:.4f}')
    return total_loss / total, correct / total

In [ ]:
@torch.no_grad()
def evaluate_epoch(model, loader, criterion):
    model.eval()

    correct, total, total_loss = 0, 0, 0.0

    pbar = tqdm(loader, desc="Validation")
    for img, labels in pbar:
        img, labels = img.to(device), labels.to(device)
        outputs = model(img)
        loss = criterion(outputs, labels)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
        total_loss += loss.item() * labels.size(0)

        pbar.set_postfix(acc=f'{100 * correct / total:.2f}%', loss=f'{total_loss / total:.4f}')
    return total_loss / total, correct / total

In [ ]:
def run_training(model, dataloader_train, dataloader_val, num_epochs, lr, criterion, checkpoint_path="best_model.pt"):
    train_loss_list = []
    val_loss_list = []
    train_acc_list = []
    val_acc_list = []
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_val_loss = float("inf")
    for epoch in range(1, num_epochs+1):

        train_loss, train_acc = train_epoch(model, dataloader_train, optimizer=optimizer, criterion=criterion, num_epochs=num_epochs, epoch=epoch)
        train_loss_list.append(train_loss)
        train_acc_list.append(train_acc)

        val_loss, val_acc = evaluate_epoch(model, dataloader_val, criterion=criterion)
        val_loss_list.append(val_loss)
        val_acc_list.append(val_acc)

        if val_loss < best_val_loss:  # keep the best checkpoint by validation loss
            best_val_loss = val_loss
            torch.save(model.state_dict(), checkpoint_path)

    return train_loss_list, train_acc_list, val_loss_list, val_acc_list

In [ ]:
# Weight the loss by inverse class frequency, since the original slices are imbalanced (few benign)
train_counts = np.bincount(targets[train_idx], minlength=len(class_names))
class_weights = torch.tensor(len(train_idx) / (len(class_names) * train_counts), dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

train_loss_list, train_acc_list, val_loss_list, val_acc_list = run_training(
    model=model,
    dataloader_train=dataloader_train,
    dataloader_val=dataloader_val,
    num_epochs=NUM_EPOCHS,
    lr=LR,
    criterion=criterion,
)

plt.subplot(2, 1, 1)
plt.title("Loss")
plt.plot(train_loss_list, label="train")
plt.plot(val_loss_list, label="val")
plt.grid()
plt.legend()

plt.subplot(2, 1, 2)
plt.title("Acc")
plt.plot(train_acc_list, label="train")
plt.plot(val_acc_list, label="val")
plt.grid()
plt.legend()

plt.tight_layout()
plt.show()

# Evaluation on the held-out test set

The test split was never used for training or for choosing the checkpoint.

In [ ]:
model.load_state_dict(torch.load("best_model.pt", map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for img, labels in dataloader_test:
        outputs = model(img.to(device))
        all_preds.extend(outputs.argmax(1).cpu().tolist())
        all_labels.extend(labels.tolist())

print(classification_report(all_labels, all_preds, target_names=class_names, digits=3))

ConfusionMatrixDisplay(confusion_matrix(all_labels, all_preds), display_labels=class_names).plot(cmap="Blues", xticks_rotation=20)
plt.show()